In [1]:
import os
import cv2 as cv
import numpy as np

Создадим папки

In [2]:
source = os.path.abspath("hw1")
folders = {color: os.path.abspath(color) for color in ("red", "yellow", "green")}

for folder in folders.values():
    os.makedirs(folder, exist_ok=True)

Пройдемся по каждому изображению. Выбор цвета светофора по максимуму пикселей из трех цветов

In [3]:
for name in os.listdir(source):
    if not name.lower().endswith((".jpg", "png")):
        continue

    path = os.path.join(source, name)
    img = cv.imdecode(np.fromfile(path, dtype=np.uint8), cv.IMREAD_COLOR)
    img_hsv = cv.cvtColor(img, cv.COLOR_BGR2HSV)

    counts = {
        "red": np.where(((img_hsv[:,:,0]<10) | (img_hsv[:,:,0]>130)) & (img_hsv[:,:,1]>130) & (img_hsv[:,:,2]>130))[0].size,
        "yellow": np.where((img_hsv[:,:,0]>10) & (img_hsv[:,:,0]<36) & (img_hsv[:,:,1]>130) & (img_hsv[:,:,2]>130))[0].size,
        "green": np.where((img_hsv[:,:,0]>36) & (img_hsv[:,:,0]<90) & (img_hsv[:,:,1]>130) & (img_hsv[:,:,2]>130))[0].size
    }

    color = max(counts, key=counts.get)
    
    base, ext = os.path.splitext(name)
    new_path = os.path.join(folders[color], name)
    number = 1
    while os.path.exists(new_path):        
        new_path = os.path.join(folders[color], f"{base}_{number}{ext}")
        number += 1

    with open(path, "rb") as original, open(new_path, "wb") as copy:
        copy.write(original.read())